# CV Eleme

Ilan metnine gore cv dosyalarini siralayacagim.


In [ ]:
import pathlib
import chromadb


### CV dosyalari


In [ ]:
files=list(pathlib.Path('data/resumes').glob('*.txt'))
print(len(files),[f.name for f in files])


In [ ]:
print(files[0].read_text()[:300])


### Veritabani


In [ ]:
col=chromadb.PersistentClient(path='./resume_db').get_or_create_collection('cv')
col.add(ids=[f.name for f in files],documents=[f.read_text() for f in files],metadatas=[{'src':f.name} for f in files])


### Sorgu


In [ ]:
job='Need an ML engineer with TensorFlow and pandas for forecasting'
res=col.query(query_texts=[job],n_results=3)
print(res['ids'][0])
print(res['documents'][0][0][:250])


### Sirala


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from IPython.display import Markdown
client=OpenAI(base_url='https://openrouter.ai/api/v1',api_key=os.getenv('OPENROUTER_API_KEY'))
def screen(job):
    hits=col.query(query_texts=[job],n_results=3)
    ctx='\n---\n'.join(hits['documents'][0])
    r=client.chat.completions.create(model='openai/gpt-oss-120b:free',messages=[{'role':'user','content':f'Resume screener. ## Rank ## Why ## Risks\nJob:{job}\n{ctx}'}])
    return Markdown(r.choices[0].message.content)
screen(job)


### Sonuc

Ada'nin cv'si tensorflow geciyor, ilk siraya geldi.
